# Stable Audio Open 1.0

https://huggingface.co/stabilityai/stable-audio-open-1.0




In [33]:
!pip uninstall -y torchao
!pip install -q torchsde -U diffusers

import torch
import soundfile as sf
from diffusers import StableAudioPipeline
from huggingface_hub import login
from IPython.display import Audio, display

login()  # paste your HF token (skip if HF_TOKEN is set in Colab secrets)

pipe = StableAudioPipeline.from_pretrained(
    "stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16
).to("cuda")

prompt = "the whispering ghosts reach out to you on the dim cold steppe of the tundra"
negative_prompt = "Low quality. Music. Musical."
generator = torch.Generator("cuda").manual_seed(0)

audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_end_in_s=13,
    num_waveforms_per_prompt=3,
    generator=generator,
).audios

sr = pipe.vae.sampling_rate
for i, a in enumerate(audio):
    sf.write(f"output_{i}.wav", a.T.float().cpu().numpy(), sr)
    display(Audio(f"output_{i}.wav"))

Output hidden; open in https://colab.research.google.com to view.

In [ ]:

prompt = "The living flesh that hates"
negative_prompt = "Low quality."
generator = torch.Generator("cuda").manual_seed(0)

audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_end_in_s=5,
    num_waveforms_per_prompt=3,
    generator=generator,
).audios

sr = pipe.vae.sampling_rate
for i, a in enumerate(audio):
    sf.write(f"output_{i}.wav", a.T.float().cpu().numpy(), sr)
    display(Audio(f"output_{i}.wav"))

# Stable Audio 3 — Small-Music

Text-to-music with Stability AI’s `stable-audio-3-small-music` (433M parameters, up to 120 s, stereo 44.1 kHz). It runs on CPU, so no GPU runtime is needed (a GPU makes it faster).

**Before running:**
1. Log in to Hugging Face and accept the terms on the [model page](https://huggingface.co/stabilityai/stable-audio-3-small-music). It’s gated, and the terms include Gemma’s (the text encoder is T5Gemma).
2. Create a read token (huggingface.co → Settings → Access Tokens) and add it as a Colab secret named `HF_TOKEN`, or paste it when `login()` asks.
3. Run the install cell. If Colab asks you to **restart the session**, do it, then continue from the next cell.

In [ ]:
# Install the Stable Audio 3 library (may replace Colab's torch with the version it pins)
!pip install -q "git+https://github.com/Stability-AI/stable-audio-3.git" soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import soundfile as sf
from huggingface_hub import login
from IPython.display import Audio, display
from stable_audio_3 import StableAudioModel

login()  # skip if HF_TOKEN is set in Colab secrets

device = "cuda" if torch.cuda.is_available() else "cpu"
model = StableAudioModel.from_pretrained("small-music", device=device)

ModuleNotFoundError: No module named 'stable_audio_3'

In [ ]:
prompt = "weird lions roaring"

out = model.generate(
    prompt=prompt,
    duration=21,    # seconds (max 120 for small-music)
    steps=8,        # default; more steps don't help the post-trained model
    seed=0,         # -1 = random
    batch_size=3,   # three variations
)
# Note: cfg_scale and negative_prompt only affect the “-base” checkpoints.

NameError: name 'model' is not defined

In [ ]:
# Normalize the output to a (batch, channels, samples) tensor plus sample rate
def unpack(out, default_sr=44100):
    if isinstance(out, (tuple, list)) and len(out) == 2:
        a, b = out
        if isinstance(a, int):
            return b, a
        if isinstance(b, int):
            return a, b
    return out, default_sr

audio, sr = unpack(out)
audio = torch.as_tensor(audio).float().cpu()
if audio.dim() == 2:          # single clip: (channels, samples)
    audio = audio.unsqueeze(0)

for i, clip in enumerate(audio):
    clip = clip / clip.abs().max().clamp(min=1e-8)   # peak normalize
    path = f"sa3_small_music_{i}.wav"
    sf.write(path, clip.T.numpy(), sr)
    print(path, f"{clip.shape[-1] / sr:.1f} s @ {sr} Hz")
    display(Audio(path))